# 01 - The data is the model  *(~5 minutes)*

> **Presenter script.** "Everyone wants to talk about model architecture.
> Almost nobody wants to talk about data cleaning. But if you hand a model
> garbage, you get a garbage model - and no clever architecture saves you.
> So let's look at some genuinely awful text and fix it."

In this notebook:

1. look at **raw, messy** text,
2. clean it in six explainable steps,
3. see **tokenization** happen character by character,
4. carve off a **validation set** - the pop quiz the model never sees.

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

In [ ]:
from minigpt import data
from minigpt.paths import DATA
from minigpt.tokenizer import CharTokenizer

import matplotlib.pyplot as plt
from minigpt.plots import use_stream_style

use_stream_style()

## 1. The raw corpus

Our starting text is a set of very simple little stories. Then we run it through
`make_messy_corpus`, which does to it what the open internet does to everything:
duplicates it, re-posts it with a tweak, mangles the character encoding, and
staples advertising to the end.

> **document** - one item in the dataset. For us, one short paragraph. Real
> corpora use one web page, one book chapter, one code file.

In [ ]:
clean_source, source_name = data.load_base_corpus()
raw_docs = data.make_messy_corpus(clean_source)

print(f"corpus source : {source_name}")
print(f"raw documents : {len(raw_docs):,}")
print(f"raw characters: {sum(len(d) for d in raw_docs):,}")

In [ ]:
def show(docs, indices, width=110):
    """Print a few documents, truncated, so they fit on a stream."""
    for i in indices:
        doc = docs[i].replace("\n", " ")
        print(f"[{i:>4}] {doc[:width]}{'...' if len(doc) > width else ''}")
        print()


print("A random handful of the RAW documents:\n")
show(raw_docs, [3, 11, 24, 57, 102, 233])

Scroll through those. You should be able to spot, by eye:

* a document that is **the same** as another one,
* a document full of **strange symbols** (`â€™`, `ï»¿`, `�`),
* a chunk of **web boilerplate** (cookie banners, copyright footers, HTML),
* something absurdly **short** (`ok.`).

Each one gets its own cleaning step below.

## 2. Broken characters (mojibake)

**Mojibake** is what you get when text saved in one character encoding is read
back in another. `'` becomes `â€™`. It is extremely common in scraped data.

Why it matters for a small model: every strange symbol becomes a **token** the
model has to learn, and our whole vocabulary is only ~80 slots. Garbage symbols
crowd out real letters.

In [ ]:
broken = [d for d in raw_docs if data.broken_char_ratio(d) > 0]
print(f"{len(broken)} of {len(raw_docs)} documents contain unexpected characters\n")
print(repr(broken[0][:180]))
print()
print(f"fraction of characters that are junk: {data.broken_char_ratio(broken[0]):.1%}")

## 3. Boilerplate

Navigation menus, cookie notices, "click here to subscribe", raw HTML. This is
the single most-repeated text on the internet, so a model trained on it happily
learns to write cookie banners instead of sentences.

In [ ]:
junk = [d for d in raw_docs if data._BOILERPLATE_PATTERNS.search(d)]
print(f"{len(junk)} boilerplate documents. Examples:\n")
for d in junk[:5]:
    print("  ", d[:100])

## 4. Length filters

Too short: no context to learn from - a two-word document teaches a next-token
model nothing. Too long: in the real world, a giant wall of repeated text is
almost always spam.

In [ ]:
normalised = data.normalize_whitespace(raw_docs)
lengths = [len(d) for d in normalised]

fig, ax = plt.subplots()
ax.hist(lengths, bins=60, color="#4c72b0")
ax.axvline(120, color="red", ls="--", label="minimum we keep (120 chars)")
ax.axvline(1200, color="red", ls="--", label="maximum we keep (1200 chars)")
ax.set_xlabel("document length (characters)")
ax.set_ylabel("how many documents")
ax.set_title("Most documents are fine; the tails are junk")
ax.legend()
plt.show()

print(f"shortest: {min(lengths)} chars   longest: {max(lengths)} chars")

## 5. Duplicates - the sneaky one

**Exact duplicates** are easy: same bytes, drop the copy.

Why it matters: a document that appears ten times is effectively trained on ten
times harder. That is precisely how models end up **reciting** training text
word for word. We will test for exactly that in notebook 03.

In [ ]:
after_exact = data.dedup_exact(normalised)
print(f"exact dedup: {len(normalised):,} -> {len(after_exact):,} "
      f"({len(normalised) - len(after_exact):,} removed)")

### Near-duplicates

Exact dedup misses the common case: the same text re-posted with different
capitalisation, extra whitespace, or `(edited)` stuck on the end.

We measure similarity with **Jaccard overlap**: chop each document into
overlapping 4-word chunks, then ask what fraction of chunks the two documents
share. `1.0` = identical, `0.0` = nothing in common. We drop anything above
`0.8`.

In [ ]:
# Build a near-duplicate on purpose so we can see the score.
import random

original = after_exact[0]
twin = "  " + original.upper().replace(" ", "  ") + "   (edited)"

score = data.jaccard(data._shingles(original), data._shingles(twin))
print("ORIGINAL:", original[:90], "...")
print("TWIN    :", twin[:90], "...")
print(f"\nJaccard overlap: {score:.2f}  -> above 0.8, so the twin gets dropped")
print(f"But are the raw strings equal?  {original == twin}  <- exact dedup would have missed it")

## 6. Run the whole pipeline

Six steps, in order, each reporting what it removed.

In [ ]:
clean_docs, steps = data.clean_corpus(raw_docs)

print(f"{'step':<26}{'kept':>8}{'removed':>10}{'% removed':>12}   why")
print("-" * 100)
for s in steps:
    print(f"{s.name:<26}{s.kept:>8,}{s.removed:>10,}{s.removed_pct:>11.1f}%   {s.why}")

About a quarter of the corpus was garbage. That is a *low* number by real-world
standards - serious web-scale pipelines throw away well over 90%.

## 7. Tokenization - turning text into numbers

A neural network does arithmetic, not spelling. So every piece of text has to
become a list of whole numbers first.

> **token** - the smallest chunk of text the model is allowed to see.
> **token ID** - the integer that stands for that chunk.
> **vocabulary** - the full list of tokens the model knows.

Big models use *sub-word* tokens (`" straw"` + `"berry"`). We use **one
character per token**, because then the entire vocabulary fits on one screen and
the model stays tiny enough to train during a coffee break.

In [ ]:
train_text = data.docs_to_text(clean_docs)
tokenizer = CharTokenizer.from_text(train_text)

print(f"vocabulary size: {tokenizer.vocab_size} tokens\n")
print("the entire vocabulary:")
print(repr("".join(tokenizer.chars[1:])))   # slot 0 is the 'unknown' token

In [ ]:
sentence = "mila found a brass key."

print(f"{'character':<12}{'token ID':>10}")
print("-" * 22)
for ch, tid in tokenizer.preview(sentence):
    shown = repr(ch) if ch in " \n" else ch
    print(f"{shown:<12}{tid:>10}")

ids = tokenizer.encode(sentence)
print(f"\nencoded : {ids}")
print(f"decoded : {tokenizer.decode(ids)!r}")
print(f"round-trip is lossless: {tokenizer.decode(ids) == sentence}")

Notice the cost of character tokens: that short sentence became **23 numbers**.
A sub-word tokenizer would have used about 6. More tokens means more work per
sentence - which is exactly the trade we are making to keep the vocabulary (and
therefore the model) small.

In [ ]:
chars = len(train_text)
print(f"training corpus: {chars:,} characters = {chars:,} tokens for us")
print(f"a real sub-word tokenizer would use roughly {chars // 4:,} tokens for the same text")

## 8. The validation split - the pop quiz

This is the most important cell in the notebook.

Before we train **anything**, we lock a slice of the data in a drawer. The model
never trains on it. Later, its score on that slice is the only honest answer to
"did it learn the language, or did it just memorise my homework?"

> **training set** - the homework. The model sees it over and over.
> **validation set** - the pop quiz. The model never sees it during training.

Two rules that people get wrong all the time:

1. **Split before you train.** Not after, not "I'll hold some out later".
2. **Split by document, not by character** - otherwise the second half of a
   sentence ends up in validation while the first half is in training, and the
   quiz is no longer honest.

In [ ]:
train_docs, val_docs = data.split_train_val(clean_docs, val_fraction=0.1, seed=0)

print(f"train: {len(train_docs):,} documents  ({sum(len(d) for d in train_docs):,} characters)")
print(f"val  : {len(val_docs):,} documents  ({sum(len(d) for d in val_docs):,} characters)")

overlap = set(train_docs) & set(val_docs)
print(f"\ndocuments appearing on BOTH sides: {len(overlap)}  <- must be 0")
assert not overlap, "leak! a validation document is also in training"

## 9. Save everything

`prepare_all()` re-runs the exact pipeline above and also builds the two extra
datasets we need later: the **recipes** corpus (notebook 04) and the
**question/answer** pairs (notebook 05). Everything lands in `data/`.

In [ ]:
summary = data.prepare_all()
print()
for path in sorted(DATA.glob("*")):
    print(f"  {path.name:24} {path.stat().st_size / 1024:8.1f} KB")

## Recap

| we did | why |
|---|---|
| dropped broken characters | junk symbols waste a tiny model's vocabulary |
| dropped boilerplate | cookie banners are not language |
| filtered by length | no context / obvious spam |
| removed exact duplicates | repeats push the model towards memorising |
| removed near duplicates | the same text with a tweak is still a repeat |
| **split off a validation set** | the only honest measure of learning |

**Next:** `02_pretrain.ipynb` - we finally train something.